# 메시지 상태 다루기 & 실행 결과 받는 방법

이 노트북에서 다루는 것:
1. **메시지 타입** (`HumanMessage` / `AIMessage` / `AnyMessage`)
2. **수동 누적** vs **`add_messages` 리듀서** — 같은 결과를 만드는 두 가지 방식 비교
3. `set_entry_point` — `START → 노드` 엣지의 다른 표기
4. **그래프 시각화** — mermaid PNG 이미지로 띄우기
5. **다양한 입력 형식** — dict 형태 메시지 (`{"role": ..., "content": ...}`)
6. **실행 결과 받는 4가지 방법**: `invoke`, `ainvoke`, `stream`, `astream`
7. **`stream_mode` 3종**: `values` / `updates` / `messages`

> 여전히 LLM 호출은 없다. 메시지 "객체" 만 들고 다닌다.

## 1. 메시지 타입

| 타입 | 의미 |
|---|---|
| `HumanMessage` | 사용자(사람) 의 메시지 |
| `AIMessage` | AI(LLM) 의 메시지 |
| `SystemMessage` | 시스템 프롬프트 |
| `ToolMessage` | 도구 호출 결과 |
| `AnyMessage` | 위의 모든 메시지 타입을 포함하는 유니온 (타입 힌트용) |

In [ ]:
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage

h = HumanMessage("안녕")
a = AIMessage("안녕하세요!")
print(type(h).__name__, h.content)
print(type(a).__name__, a.content)

## 2. 메시지 "수동" 누적 패턴 (Reducer 없음)

Reducer 를 지정하지 않으면 노드 반환값이 **덮어쓰기**다.
그래서 직접 `기존 + 새 메시지` 를 만들어 반환해야 한다.

동시에 두 가지를 보여준다:
- **`set_entry_point("node")`** = `add_edge(START, "node")` 와 같은 의미.
- `extra_field` 같은 일반 키도 State에 함께 둘 수 있다.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

class ChatStateManual(TypedDict):
    messages: list[AnyMessage]
    extra_field: int

def node(state: ChatStateManual):
    messages = state["messages"]
    new = AIMessage("안녕하세요! 무엇을 도와드릴까요?")
    # 직접 기존 messages 에 이어 붙여 돌려준다
    return {"messages": messages + [new], "extra_field": 10}

gb = StateGraph(ChatStateManual)
gb.add_node("node", node)
gb.set_entry_point("node")            # = add_edge(START, "node")
gb.add_edge("node", END)
graph_manual = gb.compile()

result = graph_manual.invoke({"messages": [HumanMessage("안녕")], "extra_field": 0})
for m in result["messages"]:
    print(f"{type(m).__name__:14s} | {m.content}")
print("extra_field:", result["extra_field"])

### 그래프 시각화 (mermaid PNG)

`get_graph().draw_mermaid_png()` 은 PNG 바이트를 반환한다.
Jupyter 환경에서 `IPython.display.Image` 로 띄울 수 있다.
(외부 mermaid 렌더링 API 를 호출하므로 네트워크가 필요하다. 오프라인이면 `draw_mermaid()` 로 텍스트만 확인.)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph_manual.get_graph().draw_mermaid_png()))
except Exception as e:
    print("PNG 렌더 실패 (네트워크/API 이슈일 수 있음):", e)
    print("\n-- mermaid 텍스트 fallback --")
    print(graph_manual.get_graph().draw_mermaid())

## 3. `add_messages` 리듀서로 단순화

위와 똑같은 동작을 더 짧게 쓸 수 있다.
노드는 그냥 **새 메시지만** 돌려주면 된다 — 합치기는 리듀서가 알아서.

또한 `add_messages` 는 다음을 자동 처리해준다:
- 단일 메시지 객체를 반환해도 리스트처럼 처리
- dict 형태 메시지(`{"role": ..., "content": ...}`) 를 적절한 메시지 객체로 변환
- 같은 `id` 가 들어오면 덮어쓰기 (update)

In [ ]:
from typing_extensions import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import START

class ChatStateAuto(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    extra_field: int

def node(state: ChatStateAuto):
    new = AIMessage("안녕하세요! 무엇을 도와드릴까요?")
    return {"messages": new, "extra_field": 10}    # 단일 메시지여도 OK

gb = StateGraph(ChatStateAuto)
gb.add_node("node", node)
gb.set_entry_point("node")
gb.add_edge("node", END)
graph_auto = gb.compile()

# dict 형태 메시지 입력도 자동 변환됨
result = graph_auto.invoke({
    "messages": [{"role": "user", "content": "안녕하세요."}],
    "extra_field": 0,
})
for m in result["messages"]:
    m.pretty_print()      # 보기 좋은 형태로 출력

## 4. 실행 결과 받는 4가지 방법

| 메서드 | 설명 |
|---|---|
| `invoke` | **동기 단발**. 결과가 나올 때까지 코드 정지 |
| `ainvoke` | **비동기 단발**. 여러 요청을 동시에 처리할 때 |
| `stream` | **동기 스트리밍**. 중간 결과를 yield |
| `astream` | **비동기 스트리밍** |

`stream` 과 `astream` 에는 `stream_mode` 옵션이 있다.

| `stream_mode` | 출력 |
|---|---|
| `"values"` | 각 단계의 **전체 State 스냅샷** |
| `"updates"` (default) | 각 단계가 만든 **부분 업데이트** dict |
| `"messages"` | 메시지 객체와 메타데이터 튜플 — LLM 토큰 스트리밍에 유용 |

### 4-1. `invoke` — 동기 단발

In [ ]:
input_message = {"role": "user", "content": "안녕하세요."}
graph_auto.invoke({"messages": [input_message], "extra_field": 0})

### 4-2. `ainvoke` — 비동기 단발

Jupyter 셀은 이미 이벤트 루프 안에서 동작하므로 `await` 를 그대로 쓸 수 있다.

In [ ]:
await graph_auto.ainvoke({"messages": [input_message], "extra_field": 0})

### 4-3. `stream` — `stream_mode="values"`

각 단계가 끝날 때마다 **현재 State 의 전체 스냅샷**을 받는다.

In [ ]:
for chunk in graph_auto.stream(
    {"messages": [input_message], "extra_field": 0},
    stream_mode="values",
):
    if chunk.get("messages"):
        chunk["messages"][-1].pretty_print()

### 4-4. `stream` — `stream_mode="updates"` (default)

각 단계가 만든 **부분 업데이트** 만 dict 로 받는다.
`{노드이름: 그 노드의 반환값}` 형태.

In [ ]:
for chunk in graph_auto.stream(
    {"messages": [input_message], "extra_field": 0},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"[{node_name}] {update}")

### 4-5. `stream` — `stream_mode="messages"`

각 메시지가 만들어질 때마다 `(메시지, 메타데이터)` 튜플을 받는다.
LLM 노드를 쓸 때 **토큰 단위 스트리밍**을 보고 싶을 때 가장 자주 쓴다.

`metadata["langgraph_node"]` 로 어느 노드에서 나온 메시지인지 알 수 있다.

In [ ]:
for chunk_msg, metadata in graph_auto.stream(
    {"messages": [input_message], "extra_field": 0},
    stream_mode="messages",
):
    print(f"[{metadata['langgraph_node']}] {type(chunk_msg).__name__}: {chunk_msg.content}")

### 4-6. `astream` — 비동기 스트리밍

`stream` 의 비동기 버전. `async for` 로 받는다.
여러 그래프를 동시에 돌리거나 FastAPI 같은 비동기 서버에서 쓸 때 유용.

In [ ]:
async for chunk_msg, metadata in graph_auto.astream(
    {"messages": [input_message], "extra_field": 0},
    stream_mode="messages",
):
    print(f"[{metadata['langgraph_node']}] {type(chunk_msg).__name__}: {chunk_msg.content}")

## 정리

- 메시지 누적은 **`add_messages` 리듀서**가 가장 깔끔. 수동 누적은 메커니즘 이해용.
- `set_entry_point("node")` = `add_edge(START, "node")`.
- `display(Image(graph.get_graph().draw_mermaid_png()))` 로 그래프 그림 확인.
- 실행 메서드 4종:
  - `invoke` / `ainvoke` — 결과 한 번에
  - `stream` / `astream` — 중간 결과 받기, **`stream_mode`** 로 출력 형식 선택

다음: `03_wiring_nodes.ipynb` — 노드/엣지 연결 패턴 (순차 / `add_sequence` / 병렬 / 조건부 병렬).